In [ ]:
import json
import numpy as np
from scipy.integrate import solve_ivp

In [ ]:
# ---- SEIR ODE system --------------------------------------------------------
def seir_rhs(t, y, beta, sigma, gamma):
    S, E, I, R = y
    N = S + E + I + R
    dS = -beta * S * I / N
    dE =  beta * S * I / N - sigma * E
    dI =  sigma * E - gamma * I
    dR =  gamma * I
    return [dS, dE, dI, dR]



In [ ]:
# ---- Choose "true" parameter values (roughly Ebola-like) --------------------
pop            = 1.0e6        # population size
R0_true        = 1.8          # basic reproduction number
latent_days    = 9.0
infectious_days= 7.0
sigma_true     = 1.0 / latent_days
gamma_true     = 1.0 / infectious_days
beta_true      = R0_true * gamma_true           # S(0) ~ N so R0 ≈ β/γ

y0  = [pop-1, 1, 0, 0]        # single initial exposure
t0, t_end, dt = 0, 240, 1.0   # ~8 months, daily steps

In [ ]:
ts   = np.arange(t0, t_end+dt, dt)
sol  = solve_ivp(seir_rhs, (t0, t_end), y0,
                 t_eval=ts,
                 args=(beta_true, sigma_true, gamma_true),
                 rtol=1e-8, atol=1e-8)

S, E, I, R = sol.y
incidence  = np.diff(R + I)    # first differences of (I+R) ≈ new infections
incidence  = np.append([1], incidence)   # keep length consistent

# Add Poisson observation noise (optional)
obs_incidence = np.random.poisson(np.maximum(incidence, 1e-3))


In [ ]:
stan_data = {
    "T"       : len(ts),
    "ts"      : ts.tolist(),
    "cases"   : obs_incidence.astype(int).tolist(),
    "S0"      : float(S[0]),
    "E0"      : float(E[0]),
    "I0"      : float(I[0]),
    "R0_init" : float(R[0])
}